# Εργαστήριο 7 — Νευρωνικά Δίκτυα, Generative AI & XAI

**Μάθημα:** Ευφυή Συστήματα και Συστήματα Υποστήριξης Αποφάσεων  
**Πανεπιστήμιο Δυτικής Αττικής (ΠΑΔΑ)**

---

## Στόχοι Εργαστηρίου

Σε αυτό το εργαστήριο θα:

- Υλοποιήσουμε χειροκίνητα έναν **Perceptron** και θα κατανοήσουμε το πρόβλημα XOR
- Εκπαιδεύσουμε ένα **MLP (Multi-Layer Perceptron)** με Keras στο Telco Churn dataset
- Εφαρμόσουμε **SHAP** για ερμηνεία των προβλέψεων του μοντέλου
- Πειραματιστούμε με **Prompt Engineering** και την παράμετρο **Temperature**

---

## Απαιτούμενες βιβλιοθήκες

```bash
pip install numpy pandas matplotlib scikit-learn tensorflow shap
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print('Βιβλιοθήκες φορτώθηκαν επιτυχώς.')

---

# Μέρος 1: Perceptron — Από τη Βιολογία στα Μαθηματικά

## 1.1 Χειροκίνητος υπολογισμός Perceptron

Ο perceptron υπολογίζει:

$$z = w_1 x_1 + w_2 x_2 + b$$
$$\hat{y} = \text{sign}(z) \quad \text{(1 αν } z > 0 \text{, αλλιώς -1)}$$

**Παράδειγμα:** Ένας απλός ταξινομητής AND gate.

In [ ]:
def perceptron_predict(x1, x2, w1, w2, b):
    """Ένας perceptron: υπολογίζει z = w·x + b και επιστρέφει sign(z)"""
    z = w1 * x1 + w2 * x2 + b
    return 1 if z > 0 else -1, z

# Βάρη για το AND gate: εκπαιδευμένα βάρη
w1, w2, b = 1.0, 1.0, -1.5

print('AND gate με Perceptron:')
print(f'{'x1':>4} {'x2':>4} {'z':>8} {'ŷ':>4} {'Αναμενόμενο':>14}')
print('-' * 40)
inputs = [(0, 0, -1), (0, 1, -1), (1, 0, -1), (1, 1, 1)]
for x1_val, x2_val, expected in inputs:
    y_hat, z = perceptron_predict(x1_val, x2_val, w1, w2, b)
    correct = '✓' if y_hat == expected else '✗'
    print(f'{x1_val:>4} {x2_val:>4} {z:>8.1f} {y_hat:>4} {expected:>8}   {correct}')

## 1.2 Το Πρόβλημα XOR — Γιατί ο Perceptron Αποτυγχάνει

Το XOR δίνει:
- (0,0) → 0, (1,1) → 0  ← κλάση A
- (0,1) → 1, (1,0) → 1  ← κλάση B

Δεν υπάρχει **ευθεία γραμμή** που να χωρίζει σωστά και τα 4 σημεία.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- AND gate (γραμμικά διαχωρίσιμο) ---
ax = axes[0]
ax.set_title('AND gate — Γραμμικά Διαχωρίσιμο', fontsize=13, fontweight='bold')
points_and = [((0,0),-1), ((0,1),-1), ((1,0),-1), ((1,1),1)]
for (x, y), label in points_and:
    color = '#3b82f6' if label == 1 else '#ef4444'
    marker = 'o' if label == 1 else 's'
    ax.scatter(x, y, c=color, marker=marker, s=200, zorder=5)
    ax.annotate(f'({x},{y})→{1 if label==1 else 0}', (x, y),
                textcoords='offset points', xytext=(10, 8), fontsize=10)

# Γραμμή διαχωρισμού
x_line = np.linspace(-0.3, 1.3, 100)
y_line = 1.5 - x_line  # w1*x + w2*y = 1.5
ax.plot(x_line, y_line, 'k--', linewidth=2, label='Γραμμή διαχωρισμού')
ax.fill_between(x_line, y_line, 1.5, alpha=0.08, color='blue')
ax.fill_between(x_line, -0.5, y_line, alpha=0.08, color='red')
ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.legend()
ax.grid(True, alpha=0.3)
patch_pos = mpatches.Patch(color='#3b82f6', label='Κλάση 1 (AND=1)')
patch_neg = mpatches.Patch(color='#ef4444', label='Κλάση -1 (AND=0)')
ax.legend(handles=[patch_pos, patch_neg], loc='upper left')

# --- XOR (μη γραμμικά διαχωρίσιμο) ---
ax = axes[1]
ax.set_title('XOR — ΜΗ Γραμμικά Διαχωρίσιμο', fontsize=13, fontweight='bold')
points_xor = [((0,0),0), ((0,1),1), ((1,0),1), ((1,1),0)]
for (x, y), label in points_xor:
    color = '#3b82f6' if label == 1 else '#ef4444'
    marker = 'o' if label == 1 else 's'
    ax.scatter(x, y, c=color, marker=marker, s=200, zorder=5)
    ax.annotate(f'({x},{y})→{label}', (x, y),
                textcoords='offset points', xytext=(10, 8), fontsize=10)

# Δοκιμή γραμμών — καμία δεν λειτουργεί
for slope, intercept, alpha in [(1, -0.5, 0.3), (0, 0.5, 0.3), (-1, 1.5, 0.3)]:
    y_try = slope * x_line + intercept
    ax.plot(x_line, y_try, 'gray', linewidth=1.5, alpha=alpha, linestyle='--')

ax.text(0.5, 0.5, '⚡ Καμία ευθεία\nδεν διαχωρίζει!',
        ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#fef3c7', edgecolor='#d97706'))
ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.grid(True, alpha=0.3)
patch_pos = mpatches.Patch(color='#3b82f6', label='Κλάση 1 (XOR=1)')
patch_neg = mpatches.Patch(color='#ef4444', label='Κλάση 0 (XOR=0)')
ax.legend(handles=[patch_pos, patch_neg], loc='upper left')

plt.tight_layout()
plt.savefig('../img/lec7/xor_linearity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Αριστερά: ευθεία γραμμή χωρίζει τέλεια το AND.')
print('Δεξιά: καμία ευθεία δεν χωρίζει το XOR → χρειαζόμαστε MLP.')

## 1.3 Εκπαίδευση Perceptron — Ο Αλγόριθμος Διόρθωσης Σφάλματος

Η κανόνας ενημέρωσης βαρών:

$$w_i \leftarrow w_i + \eta \cdot (t - \hat{y}) \cdot x_i$$

- Αν η πρόβλεψη είναι **σωστή** ($t = \hat{y}$): τα βάρη **δεν αλλάζουν**
- Αν η πρόβλεψη είναι **λάθος**: τα βάρη κινούνται προς τη σωστή κατεύθυνση

In [ ]:
def train_perceptron(X, y, learning_rate=0.1, max_epochs=20):
    """Εκπαίδευση perceptron με αλγόριθμο διόρθωσης σφάλματος."""
    n_features = X.shape[1]
    w = np.zeros(n_features)  # αρχικοποίηση βαρών στο 0
    b = 0.0
    history = []

    for epoch in range(max_epochs):
        errors = 0
        for xi, ti in zip(X, y):
            z = np.dot(w, xi) + b
            y_hat = 1 if z > 0 else -1
            if y_hat != ti:  # λάθος → διόρθωση
                w += learning_rate * (ti - y_hat) * xi
                b += learning_rate * (ti - y_hat)
                errors += 1
        history.append(errors)
        if errors == 0:
            print(f'  Σύγκλιση στην εποχή {epoch + 1}!')
            break

    return w, b, history

# Δεδομένα AND
X_and = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_and = np.array([-1, -1, -1, 1])

print('Εκπαίδευση Perceptron στο AND gate:')
w_final, b_final, history = train_perceptron(X_and, y_and)
print(f'  Τελικά βάρη: w = {w_final}, b = {b_final:.2f}')

print('\nΈλεγχος:')
for xi, ti in zip(X_and, y_and):
    z = np.dot(w_final, xi) + b_final
    y_hat = 1 if z > 0 else -1
    print(f'  x={xi}, z={z:.2f}, ŷ={y_hat}, t={ti}  {"✓" if y_hat == ti else "✗"}')

---

# Μέρος 2: MLP με Keras — Churn Prediction

## 2.1 Φόρτωση και Προεπεξεργασία Δεδομένων

In [ ]:
# Φόρτωση Telco Churn dataset
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# Προεπεξεργασία
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)

# Κωδικοποίηση κατηγορικών μεταβλητών
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

print(f'Dataset: {df.shape[0]} γραμμές, {df.shape[1]} στήλες')
print(f'Churn distribution:')
print(df['Churn'].value_counts())

In [ ]:
X = df.drop(columns=['Churn']).values
y = df['Churn'].values

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Κανονικοποίηση — κρίσιμο για τα NN!
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'\n⚠️  Σημείωση: Η κανονικοποίηση (StandardScaler) είναι κρίσιμη για τα')
print(f'    νευρωνικά δίκτυα. Χωρίς αυτή, χαρακτηριστικά με μεγάλες τιμές')
print(f'    (π.χ. TotalCharges) κυριαρχούν και η εκπαίδευση αστοχεί.')

## 2.2 Ορισμός και Εκπαίδευση MLP

Η αρχιτεκτονική μας:
```
Input (19 features)
    ↓
Dense(64) → ReLU → Dropout(0.3)
    ↓
Dense(32) → ReLU → Dropout(0.3)
    ↓
Dense(1)  → Sigmoid
    ↓
P(Churn) ∈ [0, 1]
```

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.Dropout(0.3),   # κάθε βήμα: 30% νευρώνες «σβήνουν» τυχαία
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')  # έξοδος πιθανότητας
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

n_params = model.count_params()
print(f'\nΣύνολο παραμέτρων (βαρών): {n_params:,}')

In [ ]:
# Early Stopping — σταματάμε αν το val_loss σταματήσει να βελτιώνεται
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

## 2.3 Οπτικοποίηση Εκπαίδευσης — Training vs Validation Loss

Το κλασικό διάγραμμα που δείχνει πότε εμφανίζεται **overfitting**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
ax.plot(history.history['loss'], label='Training Loss', color='#3b82f6', linewidth=2)
ax.plot(history.history['val_loss'], label='Validation Loss', color='#ef4444',
        linewidth=2, linestyle='--')
best_epoch = np.argmin(history.history['val_loss'])
ax.axvline(best_epoch, color='gray', linestyle=':', linewidth=1.5)
ax.annotate(f'Early Stop\n(epoch {best_epoch+1})',
            xy=(best_epoch, history.history['val_loss'][best_epoch]),
            xytext=(best_epoch + 3, history.history['val_loss'][best_epoch] + 0.02),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_title('Loss ανά Epoch', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Binary Cross-Entropy Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(history.history['accuracy'], label='Training Accuracy',
        color='#3b82f6', linewidth=2)
ax.plot(history.history['val_accuracy'], label='Validation Accuracy',
        color='#ef4444', linewidth=2, linestyle='--')
ax.set_title('Accuracy ανά Epoch', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Αξιολόγηση στο test set
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print('\n=== Αποτελέσματα στο Test Set ===')
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Churn', 'Churn'],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — MLP', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

# Μέρος 3: Explainable AI — SHAP

## 3.1 Γιατί χρειαζόμαστε εξηγησιμότητα;

Το MLP μας έδωσε μια πρόβλεψη — αλλά **γιατί** πρόβλεψε ότι κάποιος πελάτης θα φύγει;  
Το SHAP απαντά: **κάθε χαρακτηριστικό συνέβαλε κατά +X% ή -X%** στην τελική πρόβλεψη.

> **Additivity:** $\hat{y} = \phi_0 + \phi_1 + \phi_2 + \ldots + \phi_n$  
> Το άθροισμα κλείνει **ακριβώς** — πάντα.

In [ ]:
# Εγκατάσταση αν χρειαστεί
# !pip install shap -q

import shap

# Χρησιμοποιούμε Random Forest για SHAP (πιο γρήγορο από deep NN)
# — οι αρχές XAI ισχύουν ανεξάρτητα από τον αλγόριθμο
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
print(f'Random Forest accuracy: {rf.score(X_test, y_test):.3f}')

In [ ]:
feature_names = df.drop(columns=['Churn']).columns.tolist()

# SHAP TreeExplainer — βελτιστοποιημένος για tree-based μοντέλα
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# shap_values[1] = τιμές για κλάση 1 (Churn)
print(f'SHAP values shape: {shap_values[1].shape}')
print(f'Base value (μέση πρόβλεψη): {explainer.expected_value[1]:.3f}')

## 3.2 Global Explanation — Ποια Features Έχουν Γενικά Σημασία;

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values[1],
    X_test,
    feature_names=feature_names,
    plot_type='bar',
    show=False
)
plt.title('SHAP Global Feature Importance — Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Beeswarm plot — δείχνει κατεύθυνση (θετική/αρνητική) και κατανομή
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values[1],
    X_test,
    feature_names=feature_names,
    show=False
)
plt.title('SHAP Beeswarm — Κατεύθυνση & Μέγεθος Επίδρασης', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Κόκκινο = υψηλή τιμή feature, Μπλε = χαμηλή τιμή feature')
print('Δεξιά του 0 = αυξάνει P(Churn), Αριστερά = μειώνει P(Churn)')

## 3.3 Local Explanation — Γιατί ΑΥΤΌς ο Πελάτης Φεύγει;

Επαληθεύουμε την **additive ιδιότητα**: base value + SHAP values = τελική πρόβλεψη

In [ ]:
# Επιλέγουμε έναν πελάτη που προβλέφθηκε ως Churn
churn_indices = np.where(y_pred == 1)[0]
idx = churn_indices[0]

base_value = explainer.expected_value[1]
shap_sum = shap_values[1][idx].sum()
final_pred = base_value + shap_sum

print(f'=== Πελάτης #{idx} ===')
print(f'Βάση (μέση πρόβλεψη):  {base_value:.4f}')
print(f'Άθροισμα SHAP values:   {shap_sum:+.4f}')
print(f'Τελική πρόβλεψη:        {final_pred:.4f}')
print(f'Πρόβλεψη μοντέλου:      {rf.predict_proba(X_test[[idx]])[0,1]:.4f}')
print(f'\n✓ base + SHAP = {base_value:.4f} + {shap_sum:+.4f} = {final_pred:.4f}')
print(f'  Αυτό είναι η Additive ιδιότητα του SHAP!')

# Top 5 features που επηρεάζουν αυτόν τον πελάτη
print(f'\nTop 5 features για τον πελάτη #{idx}:')
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'Τιμή': X_test[idx],
    'SHAP': shap_values[1][idx]
}).sort_values('SHAP', key=abs, ascending=False).head(5)
print(shap_df.to_string(index=False))

In [ ]:
# Waterfall plot — ο πιο διαισθητικός τρόπος εξήγησης
shap.initjs()
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[1][idx],
        base_values=explainer.expected_value[1],
        data=X_test[idx],
        feature_names=feature_names
    )
)

---

# Μέρος 4: LLMs & Prompt Engineering

## 4.1 Η Παράμετρος Temperature

Η **θερμοκρασία (temperature)** ελέγχει πόσο «τυχαία» ή «δημιουργική» είναι η απόκριση ενός LLM:

| Temperature | Συμπεριφορά | Χρήση |
|---|---|---|
| **0.0** | Ντετερμινιστικό — πάντα την πιο πιθανή λέξη | Κώδικας, ανάλυση δεδομένων, νομικά |
| **0.5** | Ισορροπία ακρίβειας και ποικιλίας | Γενικές ερωτήσεις |
| **≥ 0.8** | Δημιουργικό — επιλέγει λιγότερο πιθανές λέξεις | Brainstorming, marketing, ιδέες |

Ας δούμε τι σημαίνει αυτό **μαθηματικά** με ένα απλό παράδειγμα.


In [ ]:
def softmax_with_temperature(logits, temperature):
    """Softmax με temperature scaling.
    Χαμηλό T → πιο sharp κατανομή (ντετερμινιστικό)
    Υψηλό T  → πιο flat κατανομή (τυχαίο)
    """
    scaled = np.array(logits) / temperature
    exp_scaled = np.exp(scaled - np.max(scaled))  # numerical stability
    return exp_scaled / exp_scaled.sum()

# Φανταστείτε ότι το μοντέλο πρέπει να επιλέξει την επόμενη λέξη
# Logits (raw scores) για 5 πιθανές λέξεις
words = ['εκπληκτικό', 'καλό', 'αξιοπρεπές', 'μέτριο', 'ασυνήθιστο']
logits = [3.0, 2.5, 1.5, 0.5, 0.2]

temperatures = [0.1, 0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)

for ax, T in zip(axes, temperatures):
    probs = softmax_with_temperature(logits, T)
    colors = ['#3b82f6' if p == probs.max() else '#93c5fd' for p in probs]
    bars = ax.bar(words, probs, color=colors)
    ax.set_title(f'Temperature = {T}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Πιθανότητα' if T == 0.1 else '')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=35)
    for bar, p in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{p:.2f}', ha='center', va='bottom', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    # Annotation
    if T == 0.1:
        ax.text(0.5, 0.85, 'Σχεδόν\nντετερμινιστικό', transform=ax.transAxes,
                ha='center', fontsize=9, color='#1e40af',
                bbox=dict(facecolor='#dbeafe', edgecolor='#3b82f6', boxstyle='round'))
    elif T == 2.0:
        ax.text(0.5, 0.85, 'Σχεδόν\nτυχαίο', transform=ax.transAxes,
                ha='center', fontsize=9, color='#991b1b',
                bbox=dict(facecolor='#fee2e2', edgecolor='#ef4444', boxstyle='round'))

plt.suptitle('Επίδραση Temperature στην Κατανομή Πιθανότητας\n(επιλογή επόμενης λέξης από LLM)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Παρατήρηση:')
print('  T=0.1: Η λέξη "εκπληκτικό" έχει σχεδόν 100% — κάθε φορά το ίδιο')
print('  T=2.0: Ακόμα και η λέξη "ασυνήθιστο" έχει αξιόλογη πιθανότητα')
print('  → Υψηλό temperature = περισσότερη δημιουργικότητα, αλλά & περισσότερα hallucinations')

## 4.2 Prompt Engineering — Σύγκριση Prompts

Εδώ θα χρησιμοποιήσουμε το **Anthropic API** (Claude) — αλλά η λογική ισχύει για κάθε LLM.

In [ ]:
# Απαιτεί: pip install anthropic
# Ορίστε το API key σας: export ANTHROPIC_API_KEY='sk-...'
# ή αλλάξτε σε OpenAI/άλλο provider

try:
    import anthropic
    import os

    client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY', 'your-key-here'))

    prompts = {
        'Κακό (vague)': 'Ανάλυσε τα δεδομένα churn.',
        'Καλό (specific)': (
            'Είσαι αναλυτής churn σε εταιρεία τηλεπικοινωνιών. '
            'Τα δεδομένα μου δείχνουν ότι το tenure και ο τύπος συμβολαίου '
            'είναι τα πιο σημαντικά features για πρόβλεψη churn. '
            'Δώσε μου 3 συγκεκριμένες στρατηγικές παρέμβασης για πελάτες '
            'με tenure < 6 μήνες και μηνιαία σύμβαση. '
            'Απάντησε σε 3 bullet points.'
        )
    }

    for label, prompt in prompts.items():
        print(f'\n{'='*60}')
        print(f'Prompt: {label}')
        print(f'{'='*60}')
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=300,
            temperature=0.3,
            messages=[{'role': 'user', 'content': prompt}]
        )
        print(response.content[0].text)

except ImportError:
    print('Η βιβλιοθήκη anthropic δεν είναι εγκατεστημένη.')
    print('Εκτελέστε: pip install anthropic')
except Exception as e:
    print(f'Σφάλμα: {e}')
    print('Ελέγξτε ότι το ANTHROPIC_API_KEY είναι σωστά ορισμένο.')

## 4.3 Επίδραση Temperature στις Απαντήσεις

Ίδιο prompt, διαφορετική temperature — παρατηρήστε πώς αλλάζει η απόκριση.

In [ ]:
try:
    import anthropic
    import os

    client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY', 'your-key-here'))

    prompt = (
        'Δώσε μου 3 ιδέες για καμπάνια διατήρησης πελατών '
        'που απειλούνται με churn στον κλάδο τηλεπικοινωνιών.'
    )

    for temp in [0.0, 1.0]:
        print(f'\n{'='*60}')
        print(f'Temperature = {temp}')
        print(f'{'='*60}')
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=250,
            temperature=temp,
            messages=[{'role': 'user', 'content': prompt}]
        )
        print(response.content[0].text)

except Exception as e:
    print(f'Σφάλμα ή API key δεν βρέθηκε: {e}')
    print('\nΠαράδειγμα αναμενόμενης διαφοράς:')
    print('  T=0.0: Τυπικές, προβλέψιμες απαντήσεις — «έκπτωση», «δωρεάν μήνας»...')
    print('  T=1.0: Πιο δημιουργικές — «gamification», «exclusive community»...')

---

# Σύνοψη

| Θέμα | Τι μάθαμε |
|---|---|
| **Perceptron** | Βιολογική αναλογία, αλγόριθμος διόρθωσης σφάλματος, αποτυχία στο XOR |
| **MLP** | Keras Sequential API, Dropout, Early Stopping, training/validation curves |
| **SHAP** | Additive εξηγήσεις, global (summary plot) vs local (waterfall) |
| **Temperature** | Μαθηματική ερμηνεία, επίδραση στη δημιουργικότητα και τα hallucinations |

## Ερωτήσεις για Εξάσκηση

1. Τι γίνεται αν αφαιρέσετε το Dropout από το MLP; Παρατηρείτε overfitting;
2. Δοκιμάστε `learning_rate=0.001` vs `0.1` — πώς επηρεάζεται η σύγκλιση;
3. Επιλέξτε έναν πελάτη που **δεν** πρόκειται να φύγει (Churn=0) και δείτε το waterfall plot. Ποια features τον κρατάνε;
4. Αλλάξτε το prompt στην ενότητα 4.2 ώστε να ζητάτε απόκριση σε συγκεκριμένη μορφή (JSON, πίνακας, bullet points). Τι αλλάζει;